# Part D — Final Submission Generation

**Project:** Academy Nova Course Cancellation Prediction  
**Group:** 51  
**Main metric:** ROC-AUC  

This notebook trains the selected final model on the full training data and generates the final Moodle submission CSV.

Important safety checks:
- preserve the original `Client_ID` order from the test set,
- do not reset or replace test IDs,
- output exactly two columns,
- make sure probabilities are between 0 and 1,
- make sure there are no missing values,
- make sure there are no duplicate `Client_ID`s.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    raise ImportError("LightGBM is required for the final model.")

# Robustly find project root by looking for the src folder
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = CURRENT_DIR
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("Could not find project root containing src/ folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.config import ID_COL, TARGET_COL, RANDOM_STATE
from src.data_loading import load_raw_data, validate_raw_data
from src.features import create_engineered_features

pd.set_option("display.max_columns", None)

Project root: /Users/mellissaghandour/Documents/GitHub/advanced-programming-project/intro-ml-final-project-academy-nova


In [3]:
train_df, test_df = load_raw_data()
validate_raw_data(train_df, test_df)

test_ids = test_df[ID_COL].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("First test IDs:")
print(test_ids.head())

Validating raw data...
Raw data validation passed.
Train shape: (63464, 29)
Test shape: (15866, 28)
Target positive rate: 0.4144
Train shape: (63464, 29)
Test shape: (15866, 28)
First test IDs:
0    62246
1    43031
2    26571
3    77694
4    22185
Name: Client_ID, dtype: int64


In [4]:
train_fe = create_engineered_features(train_df)
test_fe = create_engineered_features(test_df)

X = train_fe.drop(columns=[TARGET_COL])
y = train_fe[TARGET_COL]

X_test = test_fe.copy()

DROP_COLS = [ID_COL, "Course_Start_Date"]

X_model = X.drop(columns=[col for col in DROP_COLS if col in X.columns])
X_test_model = X_test.drop(columns=[col for col in DROP_COLS if col in X_test.columns])

print("X_model shape:", X_model.shape)
print("X_test_model shape:", X_test_model.shape)

X_model shape: (63464, 64)
X_test_model shape: (15866, 64)


In [5]:
numeric_cols = X_model.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_cols = X_model.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_cols),
        ("cat", categorical_preprocessor, categorical_cols),
    ],
    remainder="drop"
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 50
Categorical columns: 10


In [6]:
final_model = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.025,
    num_leaves=48,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.8,
    objective="binary",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_model)
])

In [7]:
final_pipeline.fit(X_model, y)

test_predictions = final_pipeline.predict_proba(X_test_model)[:, 1]

print("Prediction shape:", test_predictions.shape)
print("Min probability:", test_predictions.min())
print("Max probability:", test_predictions.max())
print("Mean probability:", test_predictions.mean())

Prediction shape: (15866,)
Min probability: 0.00036078350916947133
Max probability: 0.9997010436562291
Mean probability: 0.40263292626965064


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [8]:
submission = pd.DataFrame({
    ID_COL: test_ids,
    "Drop_Probability": test_predictions
})

submission.head()

,Client_ID,Drop_Probability
0,62246,0.178166
1,43031,0.999253
2,26571,0.122888
3,77694,0.999352
4,22185,0.151741


In [9]:
def validate_submission(submission, test_df):
    assert submission.shape[0] == test_df.shape[0], "Submission row count does not match test row count."
    assert list(submission.columns) == [ID_COL, "Drop_Probability"], "Wrong submission columns."
    assert submission[ID_COL].equals(test_df[ID_COL]), "Client_ID order changed."
    assert submission[ID_COL].isna().sum() == 0, "Missing Client_ID values."
    assert submission["Drop_Probability"].isna().sum() == 0, "Missing prediction values."
    assert submission[ID_COL].duplicated().sum() == 0, "Duplicate Client_ID values."
    assert submission["Drop_Probability"].between(0, 1).all(), "Probabilities are outside [0, 1]."

    print("✅ Submission validation passed.")
    print("Shape:", submission.shape)
    print("Columns:", submission.columns.tolist())
    print("Duplicate IDs:", submission[ID_COL].duplicated().sum())
    print("Probability range:", submission["Drop_Probability"].min(), "to", submission["Drop_Probability"].max())

validate_submission(submission, test_df)

✅ Submission validation passed.
Shape: (15866, 2)
Columns: ['Client_ID', 'Drop_Probability']
Duplicate IDs: 0
Probability range: 0.00036078350916947133 to 0.9997010436562291


In [10]:
output_path = PROJECT_ROOT / "data" / "submissions" / "Group_51_Submission_lgbm_deeper.csv"
submission.to_csv(output_path, index=False)

print("Saved submission to:", output_path)

Saved submission to: /Users/mellissaghandour/Documents/GitHub/advanced-programming-project/intro-ml-final-project-academy-nova/data/submissions/Group_51_Submission_lgbm_deeper.csv
